# S2b coverage-only CV5 smoke

Before **Save Version → Save & Run All**, attach exactly the existing Kaggle datasets `mintesnotfikir/cdd-11-30` and `hoangkhanhtung/nafnetmodel`, enable Internet only for the pinned Git clone/dependency install, and select 2×T4. This notebook never downloads dataset or checkpoint files and never enumerates or loads `CDD-11_test`. It runs the one-epoch smoke for five folds × R0/R1 and exports a lightweight audit ZIP without checkpoints.


In [ ]:
from __future__ import annotations

import datetime as dt
import hashlib
import json
import shutil
import subprocess
import sys
import traceback
from pathlib import Path

PROJECT_URL = 'https://github.com/HoangKhanhTung0111/CoT-restoration.git'
PROJECT_COMMIT = 'ac8c37374b61f46ddb893b19bc29bac366cf01de'
CDD11_ROOT = Path('/kaggle/input/datasets/mintesnotfikir/cdd-11-30')
PRETRAINED_ROOT = Path('/kaggle/input/datasets/hoangkhanhtung/nafnetmodel')
PRETRAINED = PRETRAINED_ROOT / 'NAFNet-SIDD-width32.pth'
WORK = Path('/kaggle/working/s2b_coverage_cv5')
PROJECT = WORK / 'project'
PREPARED = WORK / 'prepared'
EXPERIMENTS = WORK / 'smoke_experiments'
BUNDLE = WORK / 'bundle'
LOGS = BUNDLE / 'logs'
RUN_ID = dt.datetime.now(dt.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
ARCHIVE = Path('/kaggle/working') / f's2b_coverage_cv5_smoke_{RUN_ID}.zip'
MANIFEST = PREPARED / 'cv_manifest.json'
CACHE = PREPARED / 'generated_cache'
errors = []
stage = 'initialized'
BUNDLE.mkdir(parents=True, exist_ok=False)
LOGS.mkdir(parents=True, exist_ok=True)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def run_logged(command, name, cwd=None):
    command = [str(item) for item in command]
    print('+', ' '.join(command), flush=True)
    with (LOGS / f'{name}.log').open('w', encoding='utf-8') as log:
        process = subprocess.Popen(command, cwd=str(cwd) if cwd else None, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        code = process.wait()
    if code:
        raise RuntimeError(f'{name} failed with exit code {code}')

def record_failure():
    errors.append(f'Stage {stage} failed:\n' + traceback.format_exc())
    print(errors[-1], flush=True)


In [ ]:
try:
    stage = 'attached_input_preflight'
    assert CDD11_ROOT.is_dir(), f'Missing attached input: {CDD11_ROOT}'
    assert PRETRAINED_ROOT.is_dir(), f'Missing attached input: {PRETRAINED_ROOT}'
    assert PRETRAINED.is_file(), f'Missing SIDD32 checkpoint: {PRETRAINED}'
    gpu_names = subprocess.check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True).strip().splitlines()
    assert len(gpu_names) >= 2, f'Select Kaggle 2xT4; found {gpu_names}'
    print('Attached data:', CDD11_ROOT)
    print('Attached checkpoint:', PRETRAINED, sha256_file(PRETRAINED))
    print('GPUs:', gpu_names)

    stage = 'pinned_code_setup'
    run_logged(['git', 'clone', PROJECT_URL, PROJECT], 'clone_project')
    run_logged(['git', '-C', PROJECT, 'checkout', '--detach', PROJECT_COMMIT], 'checkout_project')
    actual = subprocess.check_output(['git', '-C', PROJECT, 'rev-parse', 'HEAD'], text=True).strip()
    assert actual == PROJECT_COMMIT
    run_logged([sys.executable, '-m', 'pip', 'install', 'einops', 'timm', 'fvcore', 'thop', 'scikit-image', 'opencv-python-headless'], 'install_dependencies')
    run_logged([sys.executable, '-m', 'unittest', 'hybrid_cot_nafnet.test_s2b_coverage', 'hybrid_cot_nafnet.test_s2b_summary', '-v'], 'unit_tests', cwd=PROJECT)
except Exception:
    record_failure()


In [ ]:
if not errors:
    try:
        stage = 'prepare_cv5_manifest_and_generated_cache'
        run_logged([sys.executable, '-u', '-m', 'hybrid_cot_nafnet.prepare_s2b_coverage', '--data-root', CDD11_ROOT, '--manifest', MANIFEST, '--cache-root', CACHE], 'prepare', cwd=PROJECT)
    except Exception:
        record_failure()


In [ ]:
if not errors:
    try:
        stage = 'cv5_r0_r1_scientific_smoke'
        run_logged([sys.executable, '-u', '-m', 'hybrid_cot_nafnet.run_s2b_coverage', '--config', PROJECT / 'configs/s2b_coverage_null_cv5_smoke_v2.json', '--data-root', CDD11_ROOT, '--manifest', MANIFEST, '--cache-root', CACHE, '--experiments-root', EXPERIMENTS, '--nproc-per-node', '2'], 'smoke', cwd=PROJECT)
    except Exception:
        record_failure()


In [ ]:
try:
    stage_before_export = stage
    for source in (MANIFEST, CACHE / 'cache_manifest.json', EXPERIMENTS / 's2b_coverage_summary.json'):
        if source.is_file():
            destination = BUNDLE / source.name
            shutil.copy2(source, destination)
    allowed = {'resolved_s2b_config.json', 'run_config.json', 'dataset_manifest.json', 'runtime_resolution.json', 'pretrained_report.json', 'git_info.json', 'environment.json', 'train_log.csv', 'memory_log.csv', 'metrics.csv', 'summary.json'}
    if EXPERIMENTS.is_dir():
        for source in EXPERIMENTS.rglob('*'):
            if source.is_file() and source.name in allowed:
                destination = BUNDLE / 'experiments' / source.relative_to(EXPERIMENTS)
                destination.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(source, destination)
    run_data = {'run_id_utc': RUN_ID, 'status': 'COMPLETE' if not errors else 'FAILED_OR_PARTIAL', 'stage': stage_before_export, 'errors': errors, 'project_commit': PROJECT_COMMIT, 'data_input': str(CDD11_ROOT), 'pretrained_input': str(PRETRAINED_ROOT), 'pretrained_sha256': sha256_file(PRETRAINED) if PRETRAINED.is_file() else None, 'dataset_or_checkpoint_downloaded': False, 'cdd11_test_opened': False}
    (BUNDLE / 'run.json').write_text(json.dumps(run_data, indent=2), encoding='utf-8')
    if ARCHIVE.exists():
        ARCHIVE.unlink()
    shutil.make_archive(str(ARCHIVE.with_suffix('')), 'zip', root_dir=BUNDLE)
    print('Download artifact:', ARCHIVE)
    print('Status:', run_data['status'])
except Exception:
    record_failure()
    raise
if errors:
    raise RuntimeError(f'S2b smoke failed; download {ARCHIVE} for audit')
